# Proposed WESAD model: Cross-Participant Generalization and Adaptive Personalization (CGAP)

This notebook tests a proposed two-branch model: a shared ExtraTrees classifier learns across the training participants, while an online SGD classifier provides participant adaptation as labeled windows arrive. Their probabilities are blended using a weight selected only with inner training-participant validation. It compares CGAP with Majority Dummy, Logistic Regression, and RBF SVM using nested participant-held-out LOSO and three seeds.

No model can be guaranteed to outperform before it is run. This experiment reports held-out results. During each replay, the model predicts a window first and then receives its true label for online updating. Treat this as a labeled-feedback scenario; real use needs a way to obtain those labels. CGAP's checkpoints are windows, not epochs.

## 1. Connect Drive and load the corrected participant CSVs

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import re
import numpy as np
import pandas as pd

DATA_DIR = Path('/content/drive/MyDrive/DataCon/Project 6 Wesad')
subject_files = sorted(p for p in DATA_DIR.glob('S*_windows.csv') if re.fullmatch(r'S\d+_windows', p.stem))
if len(subject_files) != 15:
    raise FileNotFoundError(f'Expected 15 corrected participant files in {DATA_DIR}; found {len(subject_files)}')
df = pd.concat((pd.read_csv(path) for path in subject_files), ignore_index=True)
print('Files:', len(subject_files), '| rows:', len(df), '| participants:', df.subject.nunique())
display(df.head())

Mounted at /content/drive
Files: 15 | rows: 3281 | participants: 15


,subject,window,label,label_id,acc_1_mean,acc_1_std,acc_1_min,acc_1_max,acc_2_mean,acc_2_std,...,bvp_min,bvp_max,eda_mean,eda_std,eda_min,eda_max,temp_mean,temp_std,temp_min,temp_max
0,S10,10,baseline,1,36.103125,9.492365,13.0,69.0,2.928125,2.005732,...,-247.22,199.08,0.376461,0.006709,0.367123,0.401661,33.2835,0.022085,33.25,33.33
1,S10,11,baseline,1,30.800000,3.950949,23.0,40.0,3.746875,0.576891,...,-138.13,121.99,0.452380,0.056718,0.372240,0.607608,33.3120,0.015684,33.29,33.34
2,S10,12,baseline,1,32.471875,10.737747,13.0,78.0,4.162500,2.424839,...,-264.71,229.13,0.408344,0.017809,0.377356,0.431082,33.3240,0.009165,33.31,33.33
3,S10,13,baseline,1,32.956250,12.494172,12.0,68.0,3.368750,1.910896,...,-211.67,272.99,0.378732,0.004508,0.369681,0.390148,33.3220,0.015362,33.29,33.34
4,S10,14,baseline,1,30.765625,1.596306,23.0,38.0,2.725000,1.313155,...,-29.01,23.07,0.377324,0.001820,0.368402,0.378636,33.3265,0.010851,33.31,33.34


## 2. Validate the inputs

Participant and window identifiers are used only for splitting and ordering. Labels and label IDs are never model features.

In [2]:
required = {'subject', 'window', 'label', 'label_id'}
assert required.issubset(df.columns), f'Missing columns: {required - set(df.columns)}'
FEATURES = [column for column in df.columns if column not in required]
assert FEATURES, 'No sensor feature columns found'
df[FEATURES] = df[FEATURES].apply(pd.to_numeric, errors='coerce')
assert df[FEATURES].notna().all().all(), 'Missing feature values: check that only corrected participant CSVs were loaded'
assert np.isfinite(df[FEATURES].to_numpy(dtype=float)).all(), 'Infinite feature values found'
assert df.duplicated().sum() == 0, 'Duplicate rows found'
assert df.groupby('subject').window.apply(lambda values: values.is_monotonic_increasing).all(), 'Sort participant rows by window before proceeding'
CLASSES = ['baseline', 'stress', 'amusement']
assert set(df.label.astype(str).unique()) == set(CLASSES), f'Unexpected labels: {sorted(df.label.astype(str).unique())}'
assert df.subject.nunique() == 15, f'Expected 15 participants; got {df.subject.nunique()}'
df = df.reset_index(drop=True)
X = df[FEATURES].astype(float)
y = df.label.astype(str)
groups = df.subject.astype(str)
print('Rows:', len(df), '| features:', len(FEATURES), '| classes:', y.value_counts().to_dict())

Rows: 3281 | features: 24 | classes: {'baseline': 1749, 'stress': 984, 'amusement': 548}


## 3. Define tuning and evaluation

All preprocessing and parameter selection happen inside the outer training participants. The outer held-out participant is evaluated chronologically. CGAP tuning selects the global-tree settings, online SGD regularization, and blend weight by mean inner macro-F1.

In [3]:
from itertools import product
from sklearn.model_selection import LeaveOneGroupOut, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score
import json

SEEDS = [17, 42, 2026]
OUTER = LeaveOneGroupOut()
LR_GRID = list(product([0.01, 0.1, 1.0, 10.0, 100.0], [None, 'balanced']))
SVM_GRID = list(product([0.1, 1.0, 10.0, 100.0], ['scale', 0.01, 0.1], [None, 'balanced']))
TREE_GRID = [(0.8, 2), (1.0, 5)]
SGD_ALPHA_GRID = [0.0001, 0.001]
BLEND_GRID = [0.25, 0.5, 0.75]

def score(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0),
        'macro_precision': precision_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0),
    }

def make_batch_model(name, params, seed):
    if name == 'Logistic Regression':
        c_value, class_weight = params
        classifier = LogisticRegression(C=c_value, class_weight=class_weight, max_iter=3000, random_state=seed)
    elif name == 'RBF SVM':
        c_value, gamma, class_weight = params
        classifier = SVC(C=c_value, gamma=gamma, kernel='rbf', class_weight=class_weight)
    else:
        raise ValueError(name)
    return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), classifier)

def make_cgap_global(params, seed):
    max_features, min_leaf = params
    return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), ExtraTreesClassifier(
        n_estimators=120, max_features=max_features, min_samples_leaf=min_leaf,
        class_weight='balanced', n_jobs=-1, random_state=seed))

def new_online_model(alpha, seed):
    return SGDClassifier(loss='log_loss', alpha=alpha, class_weight=None,
                        max_iter=2000, tol=1e-3, random_state=seed)

def ordered_positions(frame):
    return frame.sort_values(['subject', 'window']).index.to_numpy()

def tune_batch(name, train_positions, seed):
    grid = LR_GRID if name == 'Logistic Regression' else SVM_GRID
    inner = GroupShuffleSplit(n_splits=3, test_size=0.25, random_state=seed)
    x_train, y_train, g_train = X.loc[train_positions], y.loc[train_positions], groups.loc[train_positions]
    evaluated = []
    for params in grid:
        fold_scores = []
        for fit_local, val_local in inner.split(x_train, y_train, groups=g_train):
            model = make_batch_model(name, params, seed)
            model.fit(x_train.iloc[fit_local], y_train.iloc[fit_local])
            pred = model.predict(x_train.iloc[val_local])
            fold_scores.append(f1_score(y_train.iloc[val_local], pred, labels=CLASSES, average='macro', zero_division=0))
        evaluated.append((float(np.mean(fold_scores)), params))
    return max(evaluated, key=lambda item: item[0])

def fit_cgap(train_positions, tree_params, sgd_alpha, seed):
    train_x, train_y = X.loc[train_positions], y.loc[train_positions]
    global_model = make_cgap_global(tree_params, seed)
    global_model.fit(train_x, train_y)
    scaler = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
    train_scaled = scaler.fit_transform(train_x)
    online_model = new_online_model(sgd_alpha, seed)
    class_counts = train_y.value_counts()
    assert set(CLASSES).issubset(set(class_counts.index)), 'A training split is missing a class; change inner split settings'
    class_weight_by_label = {label: len(train_y) / (len(CLASSES) * class_counts[label]) for label in CLASSES}
    fit_weights = np.asarray([class_weight_by_label[label] for label in train_y])
    online_model.fit(train_scaled, train_y, sample_weight=fit_weights)
    return global_model, scaler, online_model, class_weight_by_label

def evaluate_cgap(global_model, scaler, online_model, class_weight_by_label, blend, frame, chunk_size=10):
    positions = ordered_positions(frame)
    actual, predicted, curve = [], [], []
    for count, position in enumerate(positions, start=1):
        raw_x = X.loc[[position]]
        online_x = scaler.transform(raw_x)
        true_label = y.loc[position]
        p_online_local = online_model.predict_proba(online_x)[0]
        p_online = np.array([p_online_local[list(online_model.classes_).index(label)] for label in CLASSES])
        p_global_local = global_model.predict_proba(raw_x)[0]
        global_classes = global_model.classes_
        p_global = np.array([p_global_local[list(global_classes).index(label)] for label in CLASSES])
        probabilities = (1.0 - blend) * p_global + blend * p_online
        pred = CLASSES[int(np.argmax(probabilities))]
        actual.append(true_label)
        predicted.append(pred)
        online_model.partial_fit(online_x, [true_label], classes=np.asarray(CLASSES),
                                 sample_weight=np.asarray([class_weight_by_label[true_label]]))
        if count % chunk_size == 0 or count == len(positions):
            curve.append({'windows_seen': count, **score(actual, predicted)})
    return actual, predicted, curve

def tune_cgap(train_positions, seed):
    train_frame = df.loc[train_positions]
    inner = GroupShuffleSplit(n_splits=3, test_size=0.25, random_state=seed)
    scored = []
    for tree_params in TREE_GRID:
        for sgd_alpha in SGD_ALPHA_GRID:
            for blend in BLEND_GRID:
                fold_scores = []
                for fit_local, val_local in inner.split(train_frame, groups=train_frame.subject.astype(str)):
                    fit_positions = train_frame.iloc[fit_local].index.to_numpy()
                    val_positions = train_frame.iloc[val_local].index.to_numpy()
                    global_model, scaler, online_model, class_weights = fit_cgap(fit_positions, tree_params, sgd_alpha, seed)
                    actual, predicted, _ = evaluate_cgap(global_model, scaler, online_model, class_weights, blend, df.loc[val_positions], chunk_size=1000000)
                    fold_scores.append(f1_score(actual, predicted, labels=CLASSES, average='macro', zero_division=0))
                scored.append((float(np.mean(fold_scores)), tree_params, sgd_alpha, blend))
    return max(scored, key=lambda item: item[0])

## 4. Run nested LOSO for three seeds

This runs parameter tuning separately inside each outer fold. It may take a while. The online branch learns from a held-out window only after making that window's prediction.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline

# ============================================================
# 1. CHECKPOINT FILES
# ============================================================
CHECKPOINT_DIR = DATA_DIR / "cgap_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_FILE = CHECKPOINT_DIR / "results_checkpoint.csv"
SELECTED_FILE = CHECKPOINT_DIR / "selected_checkpoint.csv"
CURVE_FILE = CHECKPOINT_DIR / "curve_checkpoint.csv"

results = (
    pd.read_csv(RESULTS_FILE)
    if RESULTS_FILE.exists()
    else pd.DataFrame()
)

selected_parameters = (
    pd.read_csv(SELECTED_FILE)
    if SELECTED_FILE.exists()
    else pd.DataFrame()
)

learning_curve = (
    pd.read_csv(CURVE_FILE)
    if CURVE_FILE.exists()
    else pd.DataFrame()
)

# Remove any old fold-based checkpoint columns.
# Use a separate checkpoint directory if you need to preserve
# previous fold-based results.
for frame in (results, selected_parameters, learning_curve):
    if not frame.empty and "fold" in frame.columns:
        frame.drop(columns=["fold"], inplace=True)


def save_checkpoint(results, selected_parameters, learning_curve):
    results.to_csv(RESULTS_FILE, index=False)
    selected_parameters.to_csv(SELECTED_FILE, index=False)
    learning_curve.to_csv(CURVE_FILE, index=False)
    print("Checkpoint saved.", flush=True)


# ============================================================
# 2. CREATE ONE FIXED SUBJECT-WISE TRAIN/TEST SPLIT
# ============================================================
train_pos, test_pos = next(
    OUTER.split(X, y, groups=groups)
)

train_subjects = set(groups.iloc[train_pos])
test_subjects = set(groups.iloc[test_pos])

assert len(test_subjects) == 1
assert train_subjects.isdisjoint(test_subjects)

held_out = next(iter(test_subjects))
test_frame = df.iloc[test_pos].copy()

print("Held-out subject:", held_out)
print("Training samples:", len(train_pos))
print("Testing samples:", len(test_pos))


# ============================================================
# 3. RESUME COMPLETED SEEDS
# ============================================================
completed = set()

if not results.empty and "seed" in results.columns:
    if "model" in results.columns:
        completed = set(
            results.loc[
                results["model"] == "CGAP", "seed"
            ].astype(int)
        )

print("Already completed seeds:", sorted(completed))


# ============================================================
# 4. RUN EACH SEED
# ============================================================
for seed in SEEDS:

    seed = int(seed)

    if seed in completed:
        print(f"Skipping completed seed={seed}", flush=True)
        continue

    print(f"\nRunning seed={seed}", flush=True)

    seed_records = []
    seed_selected = []
    seed_curve = []

    # --------------------------------------------------------
    # A. MAJORITY BASELINE
    # --------------------------------------------------------
    dummy = make_pipeline(
        SimpleImputer(strategy="median"),
        DummyClassifier(strategy="prior"),
    )

    dummy.fit(X.iloc[train_pos], y.iloc[train_pos])
    dummy_prediction = dummy.predict(X.iloc[test_pos])

    seed_records.append({
        "seed": seed,
        "held_out_subject": held_out,
        "model": "Majority Dummy",
        "best_inner_macro_f1": np.nan,
        "best_params": "{}",
        **score(y.iloc[test_pos], dummy_prediction),
    })

    # --------------------------------------------------------
    # B. TUNED BATCH BASELINES
    # --------------------------------------------------------
    for model_name in ["Logistic Regression", "RBF SVM"]:

        inner_score, best_params = tune_batch(
            model_name,
            train_pos,
            seed,
        )

        model = make_batch_model(
            model_name,
            best_params,
            seed,
        )

        model.fit(X.iloc[train_pos], y.iloc[train_pos])
        prediction = model.predict(X.iloc[test_pos])

        seed_records.append({
            "seed": seed,
            "held_out_subject": held_out,
            "model": model_name,
            "best_inner_macro_f1": inner_score,
            "best_params": json.dumps(best_params),
            **score(y.iloc[test_pos], prediction),
        })

        seed_selected.append({
            "seed": seed,
            "held_out_subject": held_out,
            "model": model_name,
            "inner_macro_f1": inner_score,
            "best_params": json.dumps(best_params),
        })

    # --------------------------------------------------------
    # C. TUNED CGAP
    # --------------------------------------------------------
    inner_score, tree_params, sgd_alpha, blend = tune_cgap(
        train_pos,
        seed,
    )

    (
        global_model,
        scaler,
        online_model,
        class_weights,
    ) = fit_cgap(
        train_pos,
        tree_params,
        sgd_alpha,
        seed,
    )

    actual, prediction, curve = evaluate_cgap(
        global_model,
        scaler,
        online_model,
        class_weights,
        blend,
        test_frame,
        chunk_size=10,
    )

    cgap_params = {
        "tree_max_features": tree_params[0],
        "tree_min_samples_leaf": tree_params[1],
        "sgd_alpha": sgd_alpha,
        "online_blend_weight": blend,
    }

    seed_records.append({
        "seed": seed,
        "held_out_subject": held_out,
        "model": "CGAP",
        "best_inner_macro_f1": inner_score,
        "best_params": json.dumps(cgap_params),
        **score(actual, prediction),
    })

    seed_selected.append({
        "seed": seed,
        "held_out_subject": held_out,
        "model": "CGAP",
        "inner_macro_f1": inner_score,
        "best_params": json.dumps(cgap_params),
    })

    for point in curve:
        seed_curve.append({
            "seed": seed,
            "held_out_subject": held_out,
            **point,
        })

    # --------------------------------------------------------
    # D. SAVE CHECKPOINT AFTER EACH SEED
    # --------------------------------------------------------
    results = pd.concat(
        [results, pd.DataFrame(seed_records)],
        ignore_index=True,
    )

    selected_parameters = pd.concat(
        [selected_parameters, pd.DataFrame(seed_selected)],
        ignore_index=True,
    )

    learning_curve = pd.concat(
        [learning_curve, pd.DataFrame(seed_curve)],
        ignore_index=True,
    )

    save_checkpoint(
        results,
        selected_parameters,
        learning_curve,
    )

    completed.add(seed)

    cgap_result = next(
        r for r in seed_records if r["model"] == "CGAP"
    )

    print(
        f"Finished seed={seed}; "
        f"CGAP macro-F1={cgap_result['macro_f1']:.3f}",
        flush=True,
    )


# ============================================================
# 5. FINAL SUMMARY
# ============================================================
print("\nCompleted seeds:", sorted(completed))
print("Result records:", len(results))
print("Expected records for 3 models + CGAP:", len(SEEDS) * 4)

display(results.tail(20))

Held-out subject: S10
Training samples: 3057
Testing samples: 224
Already completed seeds: []

Running seed=17


## 5. Summaries and participant-level learning curve

First average each participant's three seed results; then summarize across participants. Use mean macro-F1 and its spread for the primary comparison. Do not select a model based on the best single participant or maximum score.

## 6. Compare with the earlier tuned HAT result

The earlier HAT implementation used River and produced the values below. This notebook does not reinstall River because of the SciPy/NumPy import failures encountered in the earlier Colab runs. The HAT numbers are therefore shown as the previously reported reference, while Dummy, Logistic Regression, and RBF SVM are rerun in this notebook. Treat the HAT comparison as descriptive until all models are rerun on the exact same folds and environment.

In [ ]:
per_subject = results.groupby(['model', 'held_out_subject'], as_index=False)[metric_columns].mean()
previous_macro_f1 = {
    'Hoeffding Adaptive Tree (ADWIN), previous run': 0.560,
    'Logistic Regression, previous run': 0.457,
    'Majority Dummy, previous run': 0.232,
    'RBF SVM, previous run': 0.491,
}
comparison_rows = [{'model': model, 'macro_f1_mean': value, 'source': 'previous tuned-baseline run'}
                   for model, value in previous_macro_f1.items()]
for model in results.model.unique():
    value = float(per_subject.loc[per_subject.model == model, 'macro_f1'].mean())
    comparison_rows.append({'model': model, 'macro_f1_mean': value, 'source': 'this notebook, 3-seed LOSO'})
comparison = pd.DataFrame(comparison_rows)
display(comparison.sort_values('macro_f1_mean', ascending=False).reset_index(drop=True))

In [ ]:
metric_columns = ['accuracy', 'balanced_accuracy', 'macro_f1', 'macro_precision', 'macro_recall']
per_subject = results.groupby(['model', 'held_out_subject'], as_index=False)[metric_columns].mean()
summary = per_subject.groupby('model')[metric_columns].agg(['mean', 'std', 'min', 'max']).round(3)
display(summary)
print('Selected parameters by model and frequency:')
display(selected_parameters.groupby(['model', 'best_params']).size().rename('fold_seed_count').reset_index().sort_values(['model', 'fold_seed_count'], ascending=[True, False]))
print('CGAP cumulative metrics averaged across folds and seeds:')
display(learning_curve.groupby('windows_seen')[['accuracy', 'balanced_accuracy', 'macro_f1']].mean().round(3))

## 6. Save reproducible outputs to Drive

The saved files include raw fold-by-seed scores, participant summaries, chosen parameters, and CGAP's chronological learning curve.

In [ ]:
OUTPUT_DIR = DATA_DIR / 'cgap_proposed_model_results'
OUTPUT_DIR.mkdir(exist_ok=True)
results.to_csv(OUTPUT_DIR / 'cgap_loso_3seeds_raw.csv', index=False)
per_subject.to_csv(OUTPUT_DIR / 'cgap_loso_3seeds_per_subject.csv', index=False)
summary.to_csv(OUTPUT_DIR / 'cgap_loso_3seeds_summary.csv')
selected_parameters.to_csv(OUTPUT_DIR / 'cgap_selected_hyperparameters.csv', index=False)
learning_curve.to_csv(OUTPUT_DIR / 'cgap_online_learning_curve.csv', index=False)
print('Saved to:', OUTPUT_DIR)

## How to interpret the result

A higher CGAP mean macro-F1 than each baseline on the same held-out participants is evidence in favor of the proposed method under this labeled-feedback replay protocol. Also inspect participant-level differences and uncertainty. A small 15-person dataset cannot establish real-world superiority by itself. If you use CGAP in an interface, test the interface with users; WESAD does not contain interaction outcomes.